# 01 — Bronze Ingestion Layer
## Well Production Forecasting & Performance Analytics System

**Medallion Architecture — Bronze Layer**

This notebook implements the raw ingestion stage:
- Downloads source CSVs from GitHub (Shell hackathon data)
- Generates synthetic well production time-series using Arps decline model
- Writes raw data as Delta Lake tables with schema enforcement
- Adds metadata: `load_timestamp`, `source_file`, `data_source`
- Production data partitioned by `ingest_year`

**No business transformations** — raw values preserved.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('WellAnalytics-Bronze')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

## Step 1 — Download Source CSVs from GitHub

In [ ]:
from src.data_downloader import download_all, local_paths

downloaded = download_all(verbose=True)
paths = local_paths()
print(f'\nAvailable: {list(downloaded.keys())}')

## Step 2 — Generate Synthetic Well Production Time-Series

In [ ]:
from src.synthetic_data import save_synthetic_data
from src.spark_session import RAW_DIR

wells_path = os.path.join(RAW_DIR, 'well_metadata.csv')
synth_path = save_synthetic_data(wells_csv_path=wells_path, n_wells=300)

import pandas as pd
synth_df = pd.read_csv(synth_path)
print(f'\nSynthetic data shape: {synth_df.shape}')
print(f'Wells: {synth_df["api_number"].nunique()}')
print(f'Date range: {synth_df["production_month"].min()} to {synth_df["production_month"].max()}')
synth_df.head()

## Step 3 — Write Bronze Delta Tables

In [ ]:
from src.bronze_ingestion import run_bronze

counts = run_bronze(spark)
print('\nBronze table row counts:')
for t, n in counts.items():
    print(f'  {t:<40s} {n:>8,}')

## Step 4 — Validate Schemas and Sample Records

In [ ]:
from src.spark_session import bronze_path

for table in ['bronze_flaring', 'bronze_operator_well_counts',
              'bronze_shale_production', 'bronze_well_metadata', 'bronze_well_production']:
    df = spark.read.format('delta').load(bronze_path(table))
    print(f'\n── {table} ({df.count():,} rows) ──')
    df.printSchema()
    df.show(3, truncate=True)

## Step 5 — Partition Verification (bronze_well_production)

In [ ]:
prod_df = spark.read.format('delta').load(bronze_path('bronze_well_production'))

print('Rows per ingest_year partition:')
prod_df.groupBy('ingest_year').count().orderBy('ingest_year').show()

print('Records per oil_and_gas_group:')
prod_df.groupBy('oil_and_gas_group').count().show()